## Creación de un Verificador de Noticias en Tiempo Real

In [6]:
# Importando librerias
import os
from google import genai
from google.genai.types import (
 GenerateContentConfig,
 GoogleSearch,
 Tool)
from dotenv import load_dotenv

In [7]:
# Cargando las claves API
load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

In [16]:
# Definiendo el modelo
MODEL_ID = "gemini-2.5-flash-lite"

## Parte 1: Sin Grounding

In [19]:
# Creando la funcion la cual haga una pregunta a gemini y este responda sin busqueda web (respuesta basada en su conocimiento previo)
def pregunta_gemini(pregunta):
    response = client.models.generate_content(
        model = MODEL_ID,
        contents = pregunta,
        config = GenerateContentConfig(
            max_output_tokens=3000,
            temperature=0.7,
            top_p=0.9
        )
    )
    # Añadiendo la pregunta para que el modelo sea capaz de responder en base a los conocimientos previos (sin web)
    return response.text

# Definiendo la pregunta a realizar
pregunta = "¿Quién ha sido el ganador de la copa del rey 2026?"

# Realizando la pregunta
respuesta = pregunta_gemini(pregunta)
print(respuesta)

La Copa del Rey 2026 aún no se ha celebrado, por lo que no hay un ganador que anunciar. La competición se juega anualmente, y la edición de 2026 se llevará a cabo durante esa temporada.


Podemos ver que la respuesta no es correcta, ya que se jugó y gano la Real Sociedad, es decir, que el modelo no conoce los datos porque no ha podido buscar en la web lo cual lleva a que esta respuesta sea erronea

## Parte 2: Implementación del "Buscador Verificado"

In [20]:
def buscar_con_grounding(pregunta):
    try:
        # 3. Ejecución de la consulta
        response = client.models.generate_content(
            model=MODEL_ID,
            contents=pregunta,
            config=GenerateContentConfig(
                tools=[Tool(google_search=GoogleSearch())]
            )
        )

        # 4. Impresión del texto principal
        print(f"\n--- RESPUESTA ---\n{response.text}")

        # 5. Procesamiento detallado de fuentes (Grounding)
        metadata = response.candidates[0].grounding_metadata

        if metadata and metadata.grounding_chunks:
            print("\n--- FUENTES VERIFICADAS ---")
            for chunk in metadata.grounding_chunks:
                if chunk.web:
                    print(f"• {chunk.web.title}")
                    print(f"  Enlace: {chunk.web.uri}")
        else:
            print("\nNo se utilizaron fuentes externas específicas.")

    except Exception as e:
        print(f"Error al conectar con la API: {e}")


# Ejecución
if __name__ == "__main__":
    pregunta_usuario = "¿Quién ha sido el ganador de la copa del rey 2026?"
    buscar_con_grounding(pregunta_usuario)


--- RESPUESTA ---
La Real Sociedad se ha proclamado campeona de la Copa del Rey 2026. El equipo vasco derrotó al Atlético de Madrid en la final, que se disputó el 18 de abril de 2026 en el Estadio de La Cartuja en Sevilla. El partido terminó 2-2 tras la prórroga, y la Real Sociedad se impuso en la tanda de penaltis con un resultado de 3-4. Este es el cuarto título de Copa del Rey en la historia de la Real Sociedad.

--- FUENTES VERIFICADAS ---
• wikipedia.org
  Enlace: https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHYOMAg3AKtW-RKi74MI0bZvj2w12lmbkAe77xeeMi_dpX5FNEV2_ScaYl4HjvH7DqkeBnXr3AP9XWjjPjOI1jtewFRXBAGZrFOzA1kHKRSSbgwrwctrD1Ru1RlRE6qApmcfNZ9F-ofmPG7NQY4_Ntz
• kirolakeitb.eus
  Enlace: https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQE0ELQUVXPZgBTqIEDgYF0m3g_dUsWBGx0R0MhBpjRdsvbBpXtIYNUL14JVoePG6yc4SOrpHUS4ISpjBxcM5zVKtwWdoJ_IeAFUOWl6k7aJzPFY6eYu9X6ne6Egf4aAzazicHIHhdswkgNPU_Zddk5KkJH8ALhBalFcRZoUmkC00xlKqbZT8pQgRbk1SmogFeujlxXrs4LiTHAzmz

## Parte 3: El reto de "Anti-Alucinación"

In [29]:
def buscar_con_advertencia(pregunta):
    try:
        # 1. Ejecución de la consulta
        response = client.models.generate_content(
            model=MODEL_ID,
            contents=pregunta,
            config=GenerateContentConfig(
                tools=[Tool(google_search=GoogleSearch())]
            )
        )

        # 2. Impresión del texto principal
        print(f"\n--- RESPUESTA ---\n{response.text}")

        # 3. Procesamiento de fuentes (Grounding)
        metadata = response.candidates[0].grounding_metadata

        if metadata and metadata.grounding_chunks:
            print("\n--- FUENTES VERIFICADAS ---")
            for chunk in metadata.grounding_chunks:
                if chunk.web:
                    print(f"• {chunk.web.title}")
                    print(f"  Enlace: {chunk.web.uri}")
        else:
            print("❗Advertencia: Esta respuesta se basa en mi entrenamiento interno y no ha sido verificada en tiempo real.")

    except Exception as e:
        print(f"Error al conectar con la API: {e}")


# Ejecución
if __name__ == "__main__":
    pregunta_input = "¿Quién será el presidente de EEUU en el año 2150?"
    buscar_con_advertencia(pregunta_input)


--- RESPUESTA ---
No es posible predecir quién será el presidente de Estados Unidos en el año 2150.
❗Advertencia: Esta respuesta se basa en mi entrenamiento interno y no ha sido verificada en tiempo real.
